# libs e dados

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

In [ ]:
ds = xr.open_dataset('/content/precip_2005_2020.nc')
ds

<xarray.Dataset> Size: 531MB
Dimensions:     (valid_time: 5844, latitude: 141, longitude: 161)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 47kB 2005-01-01 ... 2020-12-31
  * latitude    (latitude) float64 1kB 5.0 4.75 4.5 4.25 ... -29.5 -29.75 -30.0
  * longitude   (longitude) float64 1kB -75.0 -74.75 -74.5 ... -35.25 -35.0
    number      int64 8B ...
    expver      (valid_time) <U4 94kB ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 531MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-09T23:34 GRIB to CDM+CF via cfgrib-0.9.1...

# data split

In [ ]:
# conjunto de treino (2005-2016)
train = ds.sel(valid_time=slice('2005', '2016'))

# conjunto de validação (2017-2018)
val = ds.sel(valid_time=slice('2017','2018'))

# conjunto de teste (2019)
test = ds.sel(valid_time=slice('2019', '2019'))

# dimensões dos conjuntos
dims = ['valid_time', 'latitude', 'longitude']
print(f"dimensões do conjunto de treino:       {tuple(train.sizes[d] for d in dims)}",
      f"dimensões do conjunto de validação:    {tuple(val.sizes[d]   for d in dims)}",
      f"dimensões do conjunto de teste:        {tuple(test.sizes[d]  for d in dims)}",
      sep="\n")

dimensões do conjunto de treino:       (4383, 141, 161)
dimensões do conjunto de validação:    (730, 141, 161)
dimensões do conjunto de teste:        (365, 141, 161)


In [ ]:
# variável de precipitação
precip_train = train['tp']
precip_val   = val['tp']
precip_test  = test['tp']

In [ ]:
# xarray → ndarray
data_train = precip_train.values
data_val   = precip_val.values
data_test  = precip_test.values

cada conjunto contém matrizes 141 × 161 \
cada matriz é a representação de uma imagem, com 141 × 161 pixels \
cada imagem é um mapa da precipitação de um dia, 5844 ao todo.

# normalização

In [ ]:
# estatisticas para normalização
mean = data_train.mean()
std  = data_train.std()

In [ ]:
# ndarray(x) → ndarray((x - μ) / δ)
def norm(data):
  return (data - mean) / std

# conjuntos normalizados
train_norm = norm(data_train)
val_norm   = norm(data_val)
test_norm  = norm(data_test)

# dimensão de canal

o tensor de entrada da rede terá o formato [N × 𝖢 × lat × lon], onde \
N = numero de amostras \
𝖢 = numero de canais \
lat = latitude em pixels \
lon = longitude em pixels \


para C, foi escolhido uma janela de 3 dias, para que a rede pudesse capturar mudanças espaço-temporais. para um dia 𝑇: \
𝖢 = [​precip(t−2)precip(t−1)precip(t)] \


 obs.: perde-se janela - 1 amostras diárias com isso: Nₙₒᵥₒ = N - 2



In [ ]:
# adiciona um canal temporal considerando 3 dias

# (5844, 141, 161) → (5842, 3, 141, 161)
def add_channel(data, window=3):
    return np.stack([data[i:i+window] for i in range(len(data)-window+1)], axis=0)

# adicionando canal temporal aos conjuntos
x_train = add_channel(train_norm)
x_val   = add_channel(val_norm)
x_test  = add_channel(test_norm)

In [ ]:
# conferindo dimensões
print("train:", x_train.shape)
print("val:", x_val.shape)
print("test:", x_test.shape)

train: (4381, 3, 141, 161)
val: (728, 3, 141, 161)
test: (363, 3, 141, 161)


# rotulação
foi utilizado uma adaptação da rótulação padronizada pelo Expert Team on Climate Change Detection and Indices, calculando os percentis de precipitação e classificando-os com base em:

se precep(t) < p90, entao é um evento normal para aquela área;  \
se precep(t) <= p99, entao é um evento intenso para aquela área; \
se precep(t) >= p99, entao é um evento extremo para aquela área. \

porém,  uma adaptação simplificada: ao inves de utilizar percentis sobre cada ponto do mapa, utiliza-se os percentis do intensidade diária (pontos máximos de cada mapa).

In [ ]:
# indicador espacial (resume a precipitação do mapa em percentis de 95)
train_indicator = np.percentile(data_train, 95, axis=(1,2))
val_indicator   = np.percentile(data_val, 95, axis=(1,2))
test_indicator  = np.percentile(data_test, 95, axis=(1,2))

In [ ]:
# remover dias perdidos pela janela
train_indicator = train_indicator[2:]
val_indicator   = val_indicator[2:]
test_indicator  = test_indicator[2:]

x_train.shape[0] == y_train.shape[0]

In [ ]:
# thresholds (somente treino)
p90 = np.percentile(train_indicator, 90)           # percentil <99: limiar eventos intensos
p99 = np.percentile(train_indicator, 99)           # percentil >99: limiar eventos extremos

# add labels aos conjuntos
def label_array(values):

    labels = np.zeros(len(values), dtype=np.int64) # percentil <90: limiar eventos normais (rótulo padrão)

    labels[(values >= p90) & (values < p99)] = 1
    labels[values >= p99] = 2

    return labels


In [ ]:
y_train = label_array(train_indicator)
y_val   = label_array(val_indicator)
y_test  = label_array(test_indicator)

In [ ]:
# balanceamento dos rótulos
print(
        np.unique(y_train, return_counts=True),
        np.unique(y_val, return_counts=True),
        np.unique(y_test, return_counts=True),
        sep="\n"
)

(array([0, 1, 2]), array([3942,  395,   44]))
(array([0, 1, 2]), array([636,  86,   6]))
(array([0, 1, 2]), array([324,  33,   6]))


# salvamento

In [ ]:
np.savez(
    "dataset.npz",
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    x_test=x_test,
    y_test=y_test
)